## Detect Depression

### GPT-5.1-thinking

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="gpt-5.1-thinking",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.1-thinking_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮: 100%|██████████| 1408/1408 [3:31:34<00:00,  9.02s/it]    

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [2]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.1-thinking_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true_original = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 将y_true中的2和3合并到1
y_true = [0 if label == 0 else 1 for label in y_true_original]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 78.76%
Precision: 47.29%
Recall: 79.56%
F1 Score: 59.32%


### GPT-5.4

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="gpt-5.4",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.4_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮: 100%|██████████| 1408/1408 [1:39:06<00:00,  4.22s/it] 

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [2]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gpt-5.4_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true_original = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 将y_true中的2和3合并到1
y_true = [0 if label == 0 else 1 for label in y_true_original]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 87.36%
Precision: 67.91%
Recall: 66.42%
F1 Score: 67.16%


### Claude-Sonnet-4-6

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="claude-sonnet-4-6",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/claude-sonnet-4-6_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮:   0%|          | 0/1408 [00:00<?, ?it/s]

第1轮:  28%|██▊       | 396/1408 [29:49<1:07:25,  4.00s/it] 

处理样本 1445999540977349888 时出错: Error code: 500 - {'error': {'message': 'Internal Server Error (500)  ', 'type': 'new_api_error', 'param': '', 'code': None}}


第1轮:  70%|██████▉   | 980/1408 [1:07:55<1:28:46, 12.45s/it]

...

处理样本 1449574568767670016 时出错: Error code: 500 - {'error': {'message': 'Internal Server Error (500)  ', 'type': 'new_api_error', 'param': '', 'code': None}}


第1轮: 100%|██████████| 1408/1408 [1:40:38<00:00,  4.29s/it]  


重试第 2/5 轮，剩余 6 条样本


第2轮: 100%|██████████| 6/6 [00:24<00:00,  4.17s/it]

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [4]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/claude-sonnet-4-6_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 86.29%
Precision: 64.94%
Recall: 64.23%
F1 Score: 64.59%


### Gemini-3.1-Pro-Preview

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="gemini-3.1-pro-preview",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gemini-3.1-pro-preview_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

重试第 1/5 轮，剩余 1408 条样本


第1轮:  37%|███▋      | 516/1408 [6:15:21<14:51:52, 59.99s/it]  

处理样本 1447942140303130112 时出错: Error code: 502


第1轮:  37%|███▋      | 517/1408 [6:16:27<15:17:58, 61.82s/it]

处理样本 1447680144026599936 时出错: Error code: 502


...

处理样本 1445493498132580096 时出错: Error code: 502


第1轮: 100%|██████████| 1408/1408 [15:20:29<00:00, 39.23s/it]   


重试第 2/5 轮，剩余 9 条样本


第2轮: 100%|██████████| 9/9 [01:42<00:00, 11.34s/it]

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [14]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/gemini-3.1-pro-preview_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 88.07%
Precision: 71.72%
Recall: 63.87%
F1 Score: 67.57%


### Kimi-K2.5

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="kimi-k2.5",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/kimi-k2.5_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [9]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/kimi-k2.5_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 80.54%
Precision: 50.00%
Recall: 87.96%
F1 Score: 63.76%


### Qwen3.5-Plus

In [ ]:
key = " "

import pandas as pd
import json, os, time
from tqdm import *
from openai import OpenAI

def identify_depression(text):
    client = OpenAI(
        base_url="https://api.linkapi.org/v1",
        api_key=key
    )
    prompt = f"""Criteria for determining if a tweet is depressed:

    - **Non-depressed Tweets**: A tweet is labeled as non-depressed if it shows joy, discusses depression generally without reflecting personal mental state, expresses casual tiredness or sadness (e.g., due to a sports team's loss), or temporary hopelessness. It may also convey any emotion other than depression.

    - **Depressed Tweets**: Express persistent hopelessness or disinterest, with symptoms like guilt, concentration problems, social withdrawal, lack of motivation, changes in weight/appetite, fatigue, reckless behavior (e.g., substance abuse), sensitivity, worthlessness, reduced productivity, low self-esteem, and excessive worry. Severe cases may involve delusions, hallucinations, or suicidal thoughts/behaviors.

    Based on the above criteria, determine if the following tweet indicates depression with a single choice option: (甲) Non-depressed Tweets, (乙) Depressed Tweets

    Tweet: "{text}"

    Please directly output option result (甲), or (乙)."""

    response = client.chat.completions.create(
        model="qwen3.5-plus",
        messages=[
            {"role": "user", "content": prompt},
        ],
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0.5,
    )
    return response.choices[0].message.content

# 确保输出目录存在
output_json_path = "/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/qwen3.5-plus_Depression_BIN.json"
os.makedirs(os.path.dirname(output_json_path), exist_ok=True)

# 如果输出文件已存在，加载已有的ID集合
success_ids = set()
if os.path.exists(output_json_path):
    try:
        with open(output_json_path, "r") as json_file:
            existing_data = json.load(json_file)
            success_ids = {item["Id"] for item in existing_data}
    except json.JSONDecodeError:
        # 文件为空或损坏，重新初始化
        with open(output_json_path, "w") as json_file:
            json.dump([], json_file)
else:
    with open(output_json_path, "w") as json_file:
        json.dump([], json_file)

# 读取数据
data = pd.read_csv('/data/zhengtianlong/Private/Depressed/Deptweet/Deptweet_test_binary.csv')
# 不打乱数据（已移除shuffle）

# 获取所有样本ID（转换为字符串以便比较）
all_ids = set(data['id'].astype(str).tolist())

max_retries = 5
current_retry = 0

# 循环直到所有样本成功或达到最大重试次数
while current_retry < max_retries and len(success_ids) < len(all_ids):
    current_retry += 1
    remaining = len(all_ids) - len(success_ids)
    print(f"重试第 {current_retry}/{max_retries} 轮，剩余 {remaining} 条样本")

    # 本轮需要处理的样本：ID不在 success_ids 中的
    remaining_data = data[~data['id'].astype(str).isin(success_ids)]

    if remaining_data.empty:
        print("所有样本已处理完成")
        break

    # 遍历剩余样本
    for _, row in tqdm(remaining_data.iterrows(), desc=f"第{current_retry}轮", total=len(remaining_data)):
        try:
            Id = str(row['id'])
            text = row['text']
            target = row['target']
            label = row['label']

            if pd.isna(text) or pd.isna(label):
                print(f"跳过缺失文本或标签的样本: {Id}")
                continue

            # 二次检查（理论上不在success_ids中）
            if Id in success_ids:
                continue

            target = str(int(target))  # 保持原处理

            # 调用API
            predict = identify_depression(text)

            result = {"Id": Id, "text": text, "predict": predict, "label": target, "target": label}

            # 写入文件（实时保存）
            with open(output_json_path, "r+") as json_file:
                file_data = json.load(json_file)
                existing_ids = {item["Id"] for item in file_data}
                if Id not in existing_ids:
                    file_data.append(result)
                    json_file.seek(0)
                    json.dump(file_data, json_file, ensure_ascii=False, indent=4)
                    json_file.truncate()  # 清除多余内容
                    success_ids.add(Id)
                else:
                    # 极少情况：文件中有但success_ids没有，则同步
                    print(f"警告: Id {Id} 已存在于文件但不在成功集合中，标记为成功")
                    success_ids.add(Id)

            time.sleep(0.5)  # 保持原延迟

        except Exception as e:
            print(f"处理样本 {Id if 'Id' in locals() else 'unknown'} 时出错: {e}")
            # 失败的不写入，留待下一轮重试
            continue

# ========== 确保最终JSON顺序与原始CSV行顺序一致 ==========
print("正在按原始CSV顺序重新排序输出文件...")
# 构建ID到原始行顺序的映射
id_to_order = {str(row['id']): idx for idx, row in data.iterrows()}

# 读取当前输出文件
with open(output_json_path, "r") as f:
    final_data = json.load(f)

# 按原始顺序排序（缺失的ID放在最后，但理论上不会出现）
final_data.sort(key=lambda x: id_to_order.get(x['Id'], float('inf')))

# 写回文件
with open(output_json_path, "w") as f:
    json.dump(final_data, f, ensure_ascii=False, indent=4)

print("排序完成！")

# 最终结果统计
if len(success_ids) == len(all_ids):
    print("所有样本处理成功！")
else:
    print(f"经过 {max_retries} 轮重试后，仍有 {len(all_ids) - len(success_ids)} 条样本未成功。")

正在按原始CSV顺序重新排序输出文件...
排序完成！
所有样本处理成功！


In [16]:
import json, random
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# 读取JSON文件
input_file_path = '/data/zhengtianlong/Private/Depressed/Close_LLM_Deptweet/qwen3.5-plus_Depression_BIN.json'
data = []

with open(input_file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# 提取预测值和真实标签
def map_label(predict):
    if "甲" in predict:
        return 0
    elif "乙" in predict:
        return 1
    else:
        return 1

y_true = [int(item["label"]) for item in data]
y_pred = [map_label(item["predict"]) for item in data]

# 计算指标
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print(f'Accuracy: {accuracy * 100:.2f}%')
print(f'Precision: {precision * 100:.2f}%')
print(f'Recall: {recall * 100:.2f}%')
print(f'F1 Score: {f1 * 100:.2f}%')


Accuracy: 86.58%
Precision: 64.41%
Recall: 69.34%
F1 Score: 66.78%
